In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scvi
import seaborn as sb
import anndata
import os

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (


### Abdominal Aortic Aneurysm (AAA)

#### 1-JD

In [2]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/1-JD',  
    var_names='gene_symbols',             
    cache=True) 

#adata.obs['dataset'] = 'Alsaigh_filtered'

In [3]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 16450 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [6]:
# include dataset column
adata.obs["dataset"] = "1_JD"
# include symptom column
adata.obs["symptoms"] = 'not stated'

In [7]:
adata

AnnData object with n_obs × n_vars = 16450 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [8]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/measure_data/1-JD/output/1_JD.h5ad")

In [9]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_2762775/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [ ]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/output_data/measure_data/1-JD/output/1_JD.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/output_data/measure_data/1-JD/output/1_JD_postR.h5ad")

In [84]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/1-JD/output/1_JD_postR.h5ad")

In [85]:
adata

AnnData object with n_obs × n_vars = 16450 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [86]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [87]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  # according to paper
sc.pp.filter_cells(adata, max_genes = 4000)  # according to paper
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author
print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10]  # according to paper
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 16450
Number of cells before MT filter: 16237
Number of cells after MT filter: 14109


In [88]:
#results_file = "../data/Plaque-datasets/Alsaigh/alsaigh_pp.h5ad"

In [89]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]

/tmp/ipykernel_2391039/986662564.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


In [90]:
adata

AnnData object with n_obs × n_vars = 14109 × 21836
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [91]:
adata.write("/home/lixiangyu/zr/Annotate/1-JD/output/1_JD_postQC.h5ad")

#### 2_ZDZJ

In [92]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/2-ZDZJ', 
    var_names='gene_symbols',             
    cache=True) 


In [93]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 17105 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [94]:
# include dataset column
adata.obs["dataset"] = "2_ZDZJ"
# include symptom column
adata.obs["symptoms"] = 'not stated'

In [95]:
adata

AnnData object with n_obs × n_vars = 17105 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [96]:
adata.write("/home/lixiangyu/zr/Annotate/2-ZDZJ/output/2_ZDZJ.h5ad")

In [97]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2391039/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [98]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/2-ZDZJ/output/2_ZDZJ.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/2-ZDZJ/output/2_ZDZJ_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 11:26:53 2025 .. Analyzing all cells
Mon Mar 10 11:26:53 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 11:27:39 2025 .... Estimating contamination
Mon Mar 10 11:27:46 2025 ...... Completed iteration: 10 | converge: 0.03671
Mon Mar 10 11:27:51 2025 ...... Completed iteration: 20 | converge: 0.02046
Mon Mar 10 11:27:57 2025 ...... Completed iteration: 30 | converge: 0.01373
Mon Mar 10 11:28:02 2025 ...... Completed iteration: 40 | converge: 0.008004
Mon Mar 10 11:28:08 2025 ...... Completed iteration: 50 | converge: 0.005419
Mon Mar 10 11:28:13 2025 ...... Completed iteration: 60 | converge: 0.004465
Mon Mar 10 11:28:19 2025 ...... Completed iteration: 70 | converge: 0.003655
Mon Mar 10 11:28:24 2025 ...... Completed iteration: 80 | converge: 0.003022
Mon Mar 10 11:28:30 2025 ...... Completed iteration: 90 | converge: 0.002622
Mon Mar 10 11:28

In [99]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/2-ZDZJ/output/2_ZDZJ_postR.h5ad")

In [100]:
adata

AnnData object with n_obs × n_vars = 17105 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [101]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [102]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 17105
Number of cells after filter: 16940
Number of cells before MT filter: 16940
Number of cells after MT filter: 14229


In [103]:
#results_file = "../data/Plaque-datasets/Alsaigh/alsaigh_pp.h5ad"

In [104]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]

/tmp/ipykernel_2391039/986662564.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


In [105]:
adata

AnnData object with n_obs × n_vars = 14229 × 21075
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [106]:
adata.write("/home/lixiangyu/zr/Annotate/2-ZDZJ/output/2_ZDZJ_postQC.h5ad")

#### AAA


In [107]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA', 
    var_names='gene_symbols',             
    cache=True) 

In [108]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 11411 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [109]:
# include dataset column
adata.obs["dataset"] = "AAA"
# include symptom column
adata.obs["symptoms"] = 'not stated'

In [110]:
adata

AnnData object with n_obs × n_vars = 11411 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [111]:
adata.write("/home/lixiangyu/zr/Annotate/AAA/output/AAA.h5ad")

In [112]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [115]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA/output/AAA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA/output/AAA_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 12:03:36 2025 .. Analyzing all cells
Mon Mar 10 12:03:36 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 12:04:06 2025 .... Estimating contamination
Mon Mar 10 12:04:11 2025 ...... Completed iteration: 10 | converge: 0.03001
Mon Mar 10 12:04:16 2025 ...... Completed iteration: 20 | converge: 0.009151
Mon Mar 10 12:04:21 2025 ...... Completed iteration: 30 | converge: 0.004638
Mon Mar 10 12:04:25 2025 ...... Completed iteration: 40 | converge: 0.002827
Mon Mar 10 12:04:30 2025 ...... Completed iteration: 50 | converge: 0.001755
Mon Mar 10 12:04:35 2025 ...... Completed iteration: 60 | converge: 0.00142
Mon Mar 10 12:04:38 2025 ...... Completed iteration: 68 | converge: 0.0008755
Mon Mar 10 12:04:38 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 1.105478 mins
----

In [116]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA/output/AAA_postR.h5ad")

In [117]:
adata

AnnData object with n_obs × n_vars = 11411 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [118]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [119]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 11411
Number of cells after filter: 10331
Number of cells before MT filter: 10331
Number of cells after MT filter: 7881


In [120]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]

/tmp/ipykernel_2391039/986662564.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


In [121]:
adata

AnnData object with n_obs × n_vars = 7881 × 21908
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [122]:
adata.write("/home/lixiangyu/zr/Annotate/AAA/output/AAA_postQC.h5ad")

#### AAA-1-3LIB

In [123]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-1-3LIB', 
    var_names='gene_symbols',             
    cache=True) 

In [124]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 13065 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [125]:
# include dataset column
adata.obs["dataset"] = "AAA-1-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'

In [126]:
adata

AnnData object with n_obs × n_vars = 13065 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [127]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-1-3LIB/output/AAA-1-3LIB.h5ad")

In [128]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [129]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-1-3LIB/output/AAA-1-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-1-3LIB/output/AAA-1-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 12:14:40 2025 .. Analyzing all cells
Mon Mar 10 12:14:40 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 12:15:15 2025 .... Estimating contamination
Mon Mar 10 12:15:20 2025 ...... Completed iteration: 10 | converge: 0.02533
Mon Mar 10 12:15:24 2025 ...... Completed iteration: 20 | converge: 0.006912
Mon Mar 10 12:15:28 2025 ...... Completed iteration: 30 | converge: 0.002538
Mon Mar 10 12:15:32 2025 ...... Completed iteration: 40 | converge: 0.001026
Mon Mar 10 12:15:32 2025 ...... Completed iteration: 41 | converge: 0.0009932
Mon Mar 10 12:15:32 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 54.92604 secs
--------------------------------------------------


In [130]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-1-3LIB/output/AAA-1-3LIB_postR.h5ad")

In [131]:
adata

AnnData object with n_obs × n_vars = 13065 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [132]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [133]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 13065
Number of cells after filter: 12757
Number of cells before MT filter: 12757
Number of cells after MT filter: 11288


In [134]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]

/tmp/ipykernel_2391039/986662564.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


In [135]:
adata

AnnData object with n_obs × n_vars = 11288 × 21258
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [136]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-1-3LIB/output/AAA-1-3LIB_postQC.h5ad")

#### AAA-2-3LIB

In [137]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-2-3LIB', 
    var_names='gene_symbols',             
    cache=True) 

In [138]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10301 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [139]:
# include dataset column
adata.obs["dataset"] = "AAA-2-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10301 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [140]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-2-3LIB/output/AAA-2-3LIB.h5ad")

In [141]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [142]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-2-3LIB/output/AAA-2-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-2-3LIB/output/AAA-2-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:04:33 2025 .. Analyzing all cells
Mon Mar 10 14:04:33 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:05:00 2025 .... Estimating contamination
Mon Mar 10 14:05:04 2025 ...... Completed iteration: 10 | converge: 0.04485
Mon Mar 10 14:05:08 2025 ...... Completed iteration: 20 | converge: 0.009539
Mon Mar 10 14:05:11 2025 ...... Completed iteration: 30 | converge: 0.003236
Mon Mar 10 14:05:15 2025 ...... Completed iteration: 40 | converge: 0.001744
Mon Mar 10 14:05:19 2025 ...... Completed iteration: 50 | converge: 0.001162
Mon Mar 10 14:05:21 2025 ...... Completed iteration: 55 | converge: 0.0009628
Mon Mar 10 14:05:21 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 50.23564 secs
--------------------------------------------------


In [143]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-2-3LIB/output/AAA-2-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10301 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [144]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [145]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10301
Number of cells after filter: 10066
Number of cells before MT filter: 10066
Number of cells after MT filter: 8993


In [146]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]

/tmp/ipykernel_2391039/986662564.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


In [147]:
adata

AnnData object with n_obs × n_vars = 8993 × 20955
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [148]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-2-3LIB/output/AAA-2-3LIB_postQC.h5ad")

#### AAA-3-3LIB

In [149]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-3-3LIB', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 11021 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [150]:
# include dataset column
adata.obs["dataset"] = "AAA-3-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 11021 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [151]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-3-3LIB/output/AAA-3-3LIB.h5ad")

In [152]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [153]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-3-3LIB/output/AAA-3-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-3-3LIB/output/AAA-3-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:07:40 2025 .. Analyzing all cells
Mon Mar 10 14:07:41 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:08:08 2025 .... Estimating contamination
Mon Mar 10 14:08:13 2025 ...... Completed iteration: 10 | converge: 0.03476
Mon Mar 10 14:08:18 2025 ...... Completed iteration: 20 | converge: 0.009501
Mon Mar 10 14:08:22 2025 ...... Completed iteration: 30 | converge: 0.003038
Mon Mar 10 14:08:26 2025 ...... Completed iteration: 40 | converge: 0.00144
Mon Mar 10 14:08:29 2025 ...... Completed iteration: 46 | converge: 0.0009426
Mon Mar 10 14:08:29 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 51.61043 secs
--------------------------------------------------


In [154]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-3-3LIB/output/AAA-3-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 11021 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [155]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [156]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 11021
Number of cells after filter: 10276
Number of cells before MT filter: 10276
Number of cells after MT filter: 8004


In [157]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8004 × 21584
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [158]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-3-3LIB/output/AAA-3-3LIB_postQC.h5ad")

#### AAA-4-3LIB

In [159]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-4-3LIB', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 14295 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [160]:
# include dataset column
adata.obs["dataset"] = "AAA-4-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 14295 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [161]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-4-3LIB/output/AAA-4-3LIB.h5ad")

In [162]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [163]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-4-3LIB/output/AAA-4-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-4-3LIB/output/AAA-4-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:11:37 2025 .. Analyzing all cells
Mon Mar 10 14:11:37 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:12:16 2025 .... Estimating contamination
Mon Mar 10 14:12:21 2025 ...... Completed iteration: 10 | converge: 0.0372
Mon Mar 10 14:12:26 2025 ...... Completed iteration: 20 | converge: 0.01358
Mon Mar 10 14:12:31 2025 ...... Completed iteration: 30 | converge: 0.007758
Mon Mar 10 14:12:35 2025 ...... Completed iteration: 40 | converge: 0.005869
Mon Mar 10 14:12:40 2025 ...... Completed iteration: 50 | converge: 0.004506
Mon Mar 10 14:12:44 2025 ...... Completed iteration: 60 | converge: 0.003525
Mon Mar 10 14:12:49 2025 ...... Completed iteration: 70 | converge: 0.00282
Mon Mar 10 14:12:53 2025 ...... Completed iteration: 80 | converge: 0.002429
Mon Mar 10 14:12:58 2025 ...... Completed iteration: 90 | converge: 0.002106
Mon Mar 10 14:13:

In [164]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-4-3LIB/output/AAA-4-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 14295 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [165]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [166]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 14295
Number of cells after filter: 14039
Number of cells before MT filter: 14039
Number of cells after MT filter: 12845


In [167]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 12845 × 21294
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [168]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-4-3LIB/output/AAA-4-3LIB_postQC.h5ad")

#### AAA-8

In [169]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-8', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 11406 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [170]:
# include dataset column
adata.obs["dataset"] = "AAA-8"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 11406 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [171]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-8/output/AAA-8.h5ad")

In [172]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [173]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-8/output/AAA-8.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-8/output/AAA-8_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:16:32 2025 .. Analyzing all cells
Mon Mar 10 14:16:32 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:17:02 2025 .... Estimating contamination
Mon Mar 10 14:17:06 2025 ...... Completed iteration: 10 | converge: 0.03292
Mon Mar 10 14:17:09 2025 ...... Completed iteration: 20 | converge: 0.01491
Mon Mar 10 14:17:12 2025 ...... Completed iteration: 30 | converge: 0.008966
Mon Mar 10 14:17:16 2025 ...... Completed iteration: 40 | converge: 0.004988
Mon Mar 10 14:17:19 2025 ...... Completed iteration: 50 | converge: 0.002729
Mon Mar 10 14:17:22 2025 ...... Completed iteration: 60 | converge: 0.00154
Mon Mar 10 14:17:25 2025 ...... Completed iteration: 67 | converge: 0.0009913
Mon Mar 10 14:17:25 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 55.62464 secs
-----

In [174]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-8/output/AAA-8_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 11406 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [175]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [176]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 11406
Number of cells after filter: 11259
Number of cells before MT filter: 11259
Number of cells after MT filter: 8668


In [177]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8668 × 20997
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [178]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-8/output/AAA-8_postQC.h5ad")

#### AAA-9

In [179]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-9', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10605 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [180]:
# include dataset column
adata.obs["dataset"] = "AAA-9"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10605 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [181]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-9/output/AAA-9.h5ad")

In [182]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [183]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-9/output/AAA-9.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-9/output/AAA-9_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:19:53 2025 .. Analyzing all cells
Mon Mar 10 14:19:53 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:20:22 2025 .... Estimating contamination
Mon Mar 10 14:20:25 2025 ...... Completed iteration: 10 | converge: 0.03029
Mon Mar 10 14:20:28 2025 ...... Completed iteration: 20 | converge: 0.01135
Mon Mar 10 14:20:31 2025 ...... Completed iteration: 30 | converge: 0.005978
Mon Mar 10 14:20:34 2025 ...... Completed iteration: 40 | converge: 0.003316
Mon Mar 10 14:20:38 2025 ...... Completed iteration: 50 | converge: 0.002069
Mon Mar 10 14:20:41 2025 ...... Completed iteration: 60 | converge: 0.001291
Mon Mar 10 14:20:42 2025 ...... Completed iteration: 65 | converge: 0.0009982
Mon Mar 10 14:20:42 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 51.05291 secs
----

In [184]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-9/output/AAA-9_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10605 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [185]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [186]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10605
Number of cells after filter: 10249
Number of cells before MT filter: 10249
Number of cells after MT filter: 8999


In [187]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8999 × 19717
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [188]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-9/output/AAA-9_postQC.h5ad")

#### AAA-D

In [189]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-D', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 12249 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [190]:
# include dataset column
adata.obs["dataset"] = "AAA-D"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 12249 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [191]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-D/output/AAA-D.h5ad")

In [192]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [193]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-D/output/AAA-D.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-D/output/AAA-D_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:23:05 2025 .. Analyzing all cells
Mon Mar 10 14:23:06 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:23:40 2025 .... Estimating contamination
Mon Mar 10 14:23:45 2025 ...... Completed iteration: 10 | converge: 0.03505
Mon Mar 10 14:23:49 2025 ...... Completed iteration: 20 | converge: 0.01228
Mon Mar 10 14:23:54 2025 ...... Completed iteration: 30 | converge: 0.007098
Mon Mar 10 14:23:58 2025 ...... Completed iteration: 40 | converge: 0.005257
Mon Mar 10 14:24:03 2025 ...... Completed iteration: 50 | converge: 0.003914
Mon Mar 10 14:24:07 2025 ...... Completed iteration: 60 | converge: 0.003023
Mon Mar 10 14:24:12 2025 ...... Completed iteration: 70 | converge: 0.002222
Mon Mar 10 14:24:16 2025 ...... Completed iteration: 80 | converge: 0.001611
Mon Mar 10 14:24:20 2025 ...... Completed iteration: 90 | converge: 0.001531
Mon Mar 10 14:2

In [194]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-D/output/AAA-D_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 12249 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [195]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [196]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 12249
Number of cells after filter: 12134
Number of cells before MT filter: 12134
Number of cells after MT filter: 10411


In [197]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 10411 × 21110
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [198]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-D/output/AAA-D_postQC.h5ad")

#### AAA-MAX

In [199]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-MAX', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8616 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [200]:
# include dataset column
adata.obs["dataset"] = "AAA-MAX"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8616 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [201]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-MAX/output/AAA-MAX.h5ad")

In [202]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [203]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-MAX/output/AAA-MAX.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-MAX/output/AAA-MAX_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:26:58 2025 .. Analyzing all cells
Mon Mar 10 14:26:58 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:27:31 2025 .... Estimating contamination
Mon Mar 10 14:27:36 2025 ...... Completed iteration: 10 | converge: 0.02799
Mon Mar 10 14:27:40 2025 ...... Completed iteration: 20 | converge: 0.00927
Mon Mar 10 14:27:44 2025 ...... Completed iteration: 30 | converge: 0.004959
Mon Mar 10 14:27:49 2025 ...... Completed iteration: 40 | converge: 0.00351
Mon Mar 10 14:27:53 2025 ...... Completed iteration: 50 | converge: 0.002266
Mon Mar 10 14:27:57 2025 ...... Completed iteration: 60 | converge: 0.001551
Mon Mar 10 14:28:02 2025 ...... Completed iteration: 70 | converge: 0.001025
Mon Mar 10 14:28:03 2025 ...... Completed iteration: 71 | converge: 0.0009928
Mon Mar 10 14:28:03 2025 .. Calculating final decontaminated matrix
------------------------

In [204]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-MAX/output/AAA-MAX_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8616 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [205]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [206]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8616
Number of cells after filter: 7711
Number of cells before MT filter: 7711
Number of cells after MT filter: 6691


In [207]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 6691 × 21367
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [208]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-MAX/output/AAA-MAX_postQC.h5ad")

#### AAA-P

In [209]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-P', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10043 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [210]:
# include dataset column
adata.obs["dataset"] = "AAA-P"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10043 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [211]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-P/output/AAA-P.h5ad")

In [212]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [213]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-P/output/AAA-P.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-P/output/AAA-P_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:30:31 2025 .. Analyzing all cells
Mon Mar 10 14:30:31 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:31:00 2025 .... Estimating contamination
Mon Mar 10 14:31:04 2025 ...... Completed iteration: 10 | converge: 0.0257
Mon Mar 10 14:31:08 2025 ...... Completed iteration: 20 | converge: 0.009209
Mon Mar 10 14:31:12 2025 ...... Completed iteration: 30 | converge: 0.006867
Mon Mar 10 14:31:15 2025 ...... Completed iteration: 40 | converge: 0.005797
Mon Mar 10 14:31:19 2025 ...... Completed iteration: 50 | converge: 0.004666
Mon Mar 10 14:31:23 2025 ...... Completed iteration: 60 | converge: 0.003625
Mon Mar 10 14:31:27 2025 ...... Completed iteration: 70 | converge: 0.002752
Mon Mar 10 14:31:31 2025 ...... Completed iteration: 80 | converge: 0.002208
Mon Mar 10 14:31:34 2025 ...... Completed iteration: 90 | converge: 0.001813
Mon Mar 10 14:3

In [214]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-P/output/AAA-P_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10043 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [215]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [216]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10043
Number of cells after filter: 9663
Number of cells before MT filter: 9663
Number of cells after MT filter: 8433


In [217]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8433 × 21059
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [218]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-P/output/AAA-P_postQC.h5ad")

#### AAA-PRO

In [219]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AAA-PRO', 
    var_names='gene_symbols',             
    cache=True) 

# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10184 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [220]:
# include dataset column
adata.obs["dataset"] = "AAA-PRO"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10184 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [221]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-PRO/output/AAA-PRO.h5ad")

In [222]:
#start with double removal and ambient RNA correction 
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro 
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate

%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


In [223]:
%%R
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/AAA-PRO/output/AAA-PRO.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/AAA-PRO/output/AAA-PRO_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 10 14:34:28 2025 .. Analyzing all cells
Mon Mar 10 14:34:28 2025 .... Generating UMAP and estimating cell types
Mon Mar 10 14:34:59 2025 .... Estimating contamination
Mon Mar 10 14:35:03 2025 ...... Completed iteration: 10 | converge: 0.03052
Mon Mar 10 14:35:07 2025 ...... Completed iteration: 20 | converge: 0.01679
Mon Mar 10 14:35:10 2025 ...... Completed iteration: 30 | converge: 0.01016
Mon Mar 10 14:35:14 2025 ...... Completed iteration: 40 | converge: 0.01108
Mon Mar 10 14:35:17 2025 ...... Completed iteration: 50 | converge: 0.01327
Mon Mar 10 14:35:21 2025 ...... Completed iteration: 60 | converge: 0.01674
Mon Mar 10 14:35:24 2025 ...... Completed iteration: 70 | converge: 0.02765
Mon Mar 10 14:35:28 2025 ...... Completed iteration: 80 | converge: 0.05518
Mon Mar 10 14:35:32 2025 ...... Completed iteration: 90 | converge: 0.04904
Mon Mar 10 14:35:35 20

In [224]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/AAA-PRO/output/AAA-PRO_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10184 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [225]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [226]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10184
Number of cells after filter: 10050
Number of cells before MT filter: 10050
Number of cells after MT filter: 8977


In [227]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2391039/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8977 × 20502
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [228]:
adata.write("/home/lixiangyu/zr/Annotate/AAA-PRO/output/AAA-PRO_postQC.h5ad")

#### RRAA

In [157]:
adata = sc.read_10x_mtx(
    './data/self_data_human/RRAA', 
    var_names='gene_symbols',             
    cache=True) 


In [158]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 15411 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [159]:
# include dataset column
adata.obs["dataset"] = "RRAA"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 15411 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [160]:
adata.write("./output_data/RRAA/output/RRAA.h5ad")

In [161]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [162]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/RRAA/output/RRAA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/RRAA/output/RRAA_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 12:25:21 2025 .. Analyzing all cells
Wed Mar 19 12:25:21 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 12:26:09 2025 .... Estimating contamination
Wed Mar 19 12:26:17 2025 ...... Completed iteration: 10 | converge: 0.02777
Wed Mar 19 12:26:24 2025 ...... Completed iteration: 20 | converge: 0.0107
Wed Mar 19 12:26:31 2025 ...... Completed iteration: 30 | converge: 0.005623
Wed Mar 19 12:26:40 2025 ...... Completed iteration: 40 | converge: 0.003398
Wed Mar 19 12:26:47 2025 ...... Completed iteration: 50 | converge: 0.002349
Wed Mar 19 12:26:53 2025 ...... Completed iteration: 60 | converge: 0.001832
Wed Mar 19 12:27:01 2025 ...... Completed iteration: 70 | converge: 0.001528
Wed Mar 19 12:27:09 2025 ...... Completed iteration: 80 | converge: 0.001451
Wed Mar 19 12:27:16 2025 ...... Completed iteration: 90 | converge: 0.001165
Wed Mar 19 12:27

In [163]:
adata = sc.read_h5ad("./output_data/RRAA/output/RRAA_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 15411 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [164]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [165]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 15411
Number of cells after filter: 15026
Number of cells before MT filter: 15026
Number of cells after MT filter: 13904


In [166]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 13904 × 21504
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [167]:
adata.write("./output_data/RRAA/output/RRAA_postQC.h5ad")

### Iliac Artery Aneurysm (IAA)

#### AIOD-3LIB  <15

In [2]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/AIOD-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [3]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 15192 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [4]:
# include dataset column
adata.obs["dataset"] = "AIOD-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 15192 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [5]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/AIOD-3LIB/output/AIOD-3LIB.h5ad")

In [6]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_3327411/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [7]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/output_data/AIOD-3LIB/output/AIOD-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/output_data/AIOD-3LIB/output/AIOD-3LIB_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [11]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/output_data/AIOD-3LIB/output/AIOD-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 15192 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [12]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [13]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 15192
Number of cells after filter: 14773
Number of cells before MT filter: 14773
Number of cells after MT filter: 12711


In [12]:
#results_file = "../data/Plaque-datasets/Alsaigh/alsaigh_pp.h5ad"

In [14]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3327411/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 12711 × 22350
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [15]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/AIOD-3LIB/output/AIOD-3LIB_postQC.h5ad")

#### IA-1-3LIB <15

In [18]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/IA-1-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [19]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 12776 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [20]:
# include dataset column
adata.obs["dataset"] = "IA-1-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 12776 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [21]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/IA-1-3LIB/output/IA-1-3LIB.h5ad")

In [22]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3327411/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [23]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/output_data/IA-1-3LIB/output/IA-1-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/output_data/IA-1-3LIB/output/IA-1-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Jul 31 17:42:29 2025 .. Analyzing all cells
Thu Jul 31 17:42:29 2025 .... Generating UMAP and estimating cell types
Thu Jul 31 17:43:00 2025 .... Estimating contamination
Thu Jul 31 17:43:04 2025 ...... Completed iteration: 10 | converge: 0.03846
Thu Jul 31 17:43:08 2025 ...... Completed iteration: 20 | converge: 0.01325
Thu Jul 31 17:43:12 2025 ...... Completed iteration: 30 | converge: 0.008695
Thu Jul 31 17:43:15 2025 ...... Completed iteration: 40 | converge: 0.006077
Thu Jul 31 17:43:19 2025 ...... Completed iteration: 50 | converge: 0.004336
Thu Jul 31 17:43:22 2025 ...... Completed iteration: 60 | converge: 0.003086
Thu Jul 31 17:43:26 2025 ...... Completed iteration: 70 | converge: 0.002196
Thu Jul 31 17:43:29 2025 ...... Completed iteration: 80 | converge: 0.001574
Thu Jul 31 17:43:33 2025 ...... Completed iteration: 90 | converge: 0.001141
Thu Jul 31 17:4

In [24]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/output_data/IA-1-3LIB/output/IA-1-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 12776 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [25]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [26]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 12776
Number of cells after filter: 12536
Number of cells before MT filter: 12536
Number of cells after MT filter: 10592


In [27]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3327411/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 10592 × 21555
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [28]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/IA-1-3LIB/output/IA-1-3LIB_postQC.h5ad")

### Atherosclerosis 

#### AS-CFA

In [33]:
adata = sc.read_10x_mtx(
    './data/self_data_human/AS-CFA', 
    var_names='gene_symbols',             
    cache=True) 


In [34]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 9268 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [ ]:
# include dataset column
adata.obs["dataset"] = "AS-CFA"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 9268 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [36]:
adata.write("./output_data/AS-CFA/output/AS-CFA.h5ad")

In [37]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2439664/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [38]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/AS-CFA/output/AS-CFA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/AS-CFA/output/AS-CFA_postR.h5ad")


Error in get(as.character(FUN), mode = "function", envir = envir) : 
  object 'as.SimpleList' of mode 'function' was not found


RInterpreterError: Failed to parse and evaluate line 'library(celda)\nlibrary(zellkonverter)\nlibrary(SummarizedExperiment)\nlibrary(scDblFinder)\nsce = readH5AD("./output_data/AS-CFA/output/AS-CFA.h5ad")\nsamples = sce$sample\nassays(sce)$counts <- assays(sce)$X\nassays(sce)$X <- NULL\nsce1 <- scDblFinder(sce, samples=samples)\nsce2 <- decontX(sce1, batch = samples)\nsce_adata <- writeH5AD(sce2, file="./output_data/AS-CFA/output/AS-CFA_postR.h5ad")\n'.
R error message: 'Error in get(as.character(FUN), mode = "function", envir = envir) : \n  object \'as.SimpleList\' of mode \'function\' was not found'

In [19]:
adata = sc.read_h5ad("./output_data/AS-CFA/output/AS-CFA_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 9268 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [20]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [21]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 9268
Number of cells after filter: 9002
Number of cells before MT filter: 9002
Number of cells after MT filter: 8209


In [22]:
#results_file = "../data/Plaque-datasets/Alsaigh/alsaigh_pp.h5ad"

In [23]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2439664/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8209 × 20836
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [24]:
adata.write("./output_data/AS-CFA/output/AS-CFA_postQC.h5ad")

#### AS-FA

In [40]:
adata = sc.read_10x_mtx(
    './data/self_data_human/AS-FA', 
    var_names='gene_symbols',             
    cache=True) 


In [41]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 14198 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [42]:
# include dataset column
adata.obs["dataset"] = "AS-FA"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 14198 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [43]:
adata.write("./output_data/AS-FA/output/AS-FA.h5ad")

In [44]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_2439664/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [45]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/AS-FA/output/AS-FA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/AS-FA/output/AS-FA_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 17 16:35:17 2025 .. Analyzing all cells
Mon Mar 17 16:35:17 2025 .... Generating UMAP and estimating cell types
Mon Mar 17 16:36:00 2025 .... Estimating contamination
Mon Mar 17 16:36:06 2025 ...... Completed iteration: 10 | converge: 0.02318
Mon Mar 17 16:36:12 2025 ...... Completed iteration: 20 | converge: 0.008106
Mon Mar 17 16:36:17 2025 ...... Completed iteration: 30 | converge: 0.004525
Mon Mar 17 16:36:23 2025 ...... Completed iteration: 40 | converge: 0.002966
Mon Mar 17 16:36:28 2025 ...... Completed iteration: 50 | converge: 0.001994
Mon Mar 17 16:36:34 2025 ...... Completed iteration: 60 | converge: 0.001516
Mon Mar 17 16:36:40 2025 ...... Completed iteration: 70 | converge: 0.001057
Mon Mar 17 16:36:41 2025 ...... Completed iteration: 71 | converge: 0.0009713
Mon Mar 17 16:36:41 2025 .. Calculating final decontaminated matrix
----------------------

In [46]:
adata = sc.read_h5ad("./output_data/AS-FA/output/AS-FA_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 14198 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [47]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [48]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 14198
Number of cells after filter: 13706
Number of cells before MT filter: 13706
Number of cells after MT filter: 11978


In [49]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2439664/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 11978 × 21029
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [50]:
adata.write("./output_data/AS-FA/output/AS-FA_postQC.h5ad")

#### AS-POP

In [51]:
adata = sc.read_10x_mtx(
    './data/self_data_human/AS-POP', 
    var_names='gene_symbols',             
    cache=True) 


In [52]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 9196 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [53]:
# include dataset column
adata.obs["dataset"] = "AS-POP"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 9196 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [54]:
adata.write("./output_data/AS-POP/output/AS-POP.h5ad")

In [55]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2439664/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [56]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/AS-POP/output/AS-POP.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/AS-POP/output/AS-POP_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 17 16:39:09 2025 .. Analyzing all cells
Mon Mar 17 16:39:09 2025 .... Generating UMAP and estimating cell types
Mon Mar 17 16:39:43 2025 .... Estimating contamination
Mon Mar 17 16:39:49 2025 ...... Completed iteration: 10 | converge: 0.03023
Mon Mar 17 16:39:53 2025 ...... Completed iteration: 20 | converge: 0.01015
Mon Mar 17 16:39:58 2025 ...... Completed iteration: 30 | converge: 0.005802
Mon Mar 17 16:40:03 2025 ...... Completed iteration: 40 | converge: 0.003344
Mon Mar 17 16:40:09 2025 ...... Completed iteration: 50 | converge: 0.002083
Mon Mar 17 16:40:14 2025 ...... Completed iteration: 60 | converge: 0.001424
Mon Mar 17 16:40:19 2025 ...... Completed iteration: 70 | converge: 0.001059
Mon Mar 17 16:40:20 2025 ...... Completed iteration: 72 | converge: 0.0009808
Mon Mar 17 16:40:20 2025 .. Calculating final decontaminated matrix
-----------------------

In [57]:
adata = sc.read_h5ad("./output_data/AS-POP/output/AS-POP_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 9196 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [58]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [59]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 9196
Number of cells after filter: 8113
Number of cells before MT filter: 8113
Number of cells after MT filter: 6403


In [60]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2439664/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 6403 × 21955
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [61]:
adata.write("./output_data/AS-POP/output/AS-POP_postQC.h5ad")

#### CFA-BWD

In [3]:
adata = sc.read_10x_mtx(
    './data/self_data_human/CFA-BWD', 
    var_names='gene_symbols',             
    cache=True) 


In [4]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8279 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [5]:
# include dataset column
adata.obs["dataset"] = "CFA-BWD"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8279 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [6]:
adata.write("./output_data/CFA-BWD/output/CFA-BWD.h5ad")

In [7]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/numpy2ri.py:252: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2938674/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [8]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/CFA-BWD/output/CFA-BWD.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/CFA-BWD/output/CFA-BWD_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [9]:
adata = sc.read_h5ad("./output_data/CFA-BWD/output/CFA-BWD_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8279 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [10]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [11]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8279
Number of cells after filter: 7906
Number of cells before MT filter: 7906
Number of cells after MT filter: 7318


In [12]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2938674/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 7318 × 20560
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [13]:
adata.write("./output_data/CFA-BWD/output/CFA-BWD_postQC.h5ad")

#### CFA-PLA

In [14]:
adata = sc.read_10x_mtx(
    './data/self_data_human/CFA-PLA', 
    var_names='gene_symbols',             
    cache=True) 


In [15]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8039 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [16]:
# include dataset column
adata.obs["dataset"] = "CFA-PLA"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8039 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [17]:
adata.write("./output_data/CFA-PLA/output/CFA-PLA.h5ad")

In [18]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2938674/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [19]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/CFA-PLA/output/CFA-PLA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/CFA-PLA/output/CFA-PLA_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Mar 18 11:30:17 2025 .. Analyzing all cells
Tue Mar 18 11:30:17 2025 .... Generating UMAP and estimating cell types
Tue Mar 18 11:30:44 2025 .... Estimating contamination
Tue Mar 18 11:30:48 2025 ...... Completed iteration: 10 | converge: 0.02373
Tue Mar 18 11:30:51 2025 ...... Completed iteration: 20 | converge: 0.009943
Tue Mar 18 11:30:54 2025 ...... Completed iteration: 30 | converge: 0.005942
Tue Mar 18 11:30:56 2025 ...... Completed iteration: 40 | converge: 0.004
Tue Mar 18 11:30:59 2025 ...... Completed iteration: 50 | converge: 0.002851
Tue Mar 18 11:31:02 2025 ...... Completed iteration: 60 | converge: 0.002218
Tue Mar 18 11:31:05 2025 ...... Completed iteration: 70 | converge: 0.001727
Tue Mar 18 11:31:08 2025 ...... Completed iteration: 80 | converge: 0.001468
Tue Mar 18 11:31:11 2025 ...... Completed iteration: 90 | converge: 0.001079
Tue Mar 18 11:31:

In [20]:
adata = sc.read_h5ad("./output_data/CFA-PLA/output/CFA-PLA_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8039 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [21]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [22]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8039
Number of cells after filter: 7882
Number of cells before MT filter: 7882
Number of cells after MT filter: 7730


In [23]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2938674/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 7730 × 20704
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [24]:
adata.write("./output_data/CFA-PLA/output/CFA-PLA_postQC.h5ad")

#### MXT_matrix  min_cells=1 <25

In [40]:
adata = sc.read_10x_mtx(
    './data/self_data_human/MXT_matrix', 
    var_names='gene_symbols',             
    cache=True) 


In [41]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 9020 × 58336
    obs: 'sample'
    var: 'gene_ids'

In [42]:
# include dataset column
adata.obs["dataset"] = "MXT_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 9020 × 58336
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [43]:
adata.write("./output_data/MXT_matrix/output/MXT_matrix.h5ad")

In [44]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_1490241/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [45]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/MXT_matrix/output/MXT_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/MXT_matrix/output/MXT_matrix_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Mon Mar 24 15:41:01 2025 .. Analyzing all cells
Mon Mar 24 15:41:01 2025 .... Generating UMAP and estimating cell types
Mon Mar 24 15:41:37 2025 .... Estimating contamination
Mon Mar 24 15:41:42 2025 ...... Completed iteration: 10 | converge: 0.02694
Mon Mar 24 15:41:46 2025 ...... Completed iteration: 20 | converge: 0.008398
Mon Mar 24 15:41:50 2025 ...... Completed iteration: 30 | converge: 0.004672
Mon Mar 24 15:41:54 2025 ...... Completed iteration: 40 | converge: 0.002895
Mon Mar 24 15:41:58 2025 ...... Completed iteration: 50 | converge: 0.00195
Mon Mar 24 15:42:02 2025 ...... Completed iteration: 60 | converge: 0.002769
Mon Mar 24 15:42:06 2025 ...... Completed iteration: 70 | converge: 0.001158
Mon Mar 24 15:42:10 2025 ...... Completed iteration: 80 | converge: 0.001143
Mon Mar 24 15:42:14 2025 ...... Completed iteration: 90 | converge: 0.001207
Mon Mar 24 15:4

In [46]:
adata = sc.read_h5ad("./output_data/MXT_matrix/output/MXT_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 9020 × 58336
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [47]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [48]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=1)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 25] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 9020
Number of cells after filter: 8920
Number of cells before MT filter: 8920
Number of cells after MT filter: 6971


In [49]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_1490241/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 6971 × 28884
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [50]:
adata.write("./output_data/MXT_matrix/output/MXT_matrix_postQC.h5ad")

#### SFA-8

In [170]:
adata = sc.read_10x_mtx(
    './data/self_data_human/SFA-8', 
    var_names='gene_symbols',             
    cache=True) 


In [171]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8889 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [172]:
# include dataset column
adata.obs["dataset"] = "SFA-8"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8889 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [173]:
adata.write("./output_data/SFA-8/output/SFA-8.h5ad")

In [174]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [175]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/SFA-8/output/SFA-8.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/SFA-8/output/SFA-8_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 12:32:20 2025 .. Analyzing all cells
Wed Mar 19 12:32:20 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 12:32:59 2025 .... Estimating contamination
Wed Mar 19 12:33:05 2025 ...... Completed iteration: 10 | converge: 0.0301
Wed Mar 19 12:33:11 2025 ...... Completed iteration: 20 | converge: 0.0226
Wed Mar 19 12:33:17 2025 ...... Completed iteration: 30 | converge: 0.01321
Wed Mar 19 12:33:23 2025 ...... Completed iteration: 40 | converge: 0.01098
Wed Mar 19 12:33:29 2025 ...... Completed iteration: 50 | converge: 0.009537
Wed Mar 19 12:33:35 2025 ...... Completed iteration: 60 | converge: 0.01038
Wed Mar 19 12:33:41 2025 ...... Completed iteration: 70 | converge: 0.01071
Wed Mar 19 12:33:46 2025 ...... Completed iteration: 80 | converge: 0.009353
Wed Mar 19 12:33:51 2025 ...... Completed iteration: 90 | converge: 0.008158
Wed Mar 19 12:33:56 2

In [176]:
adata = sc.read_h5ad("./output_data/SFA-8/output/SFA-8_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8889 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [177]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [178]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8889
Number of cells after filter: 8018
Number of cells before MT filter: 8018
Number of cells after MT filter: 7567


In [179]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 7567 × 21144
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [180]:
adata.write("./output_data/SFA-8/output/SFA-8_postQC.h5ad")

### Carotid Body Tumor (CBT)

#### CBT

In [29]:
adata = sc.read_10x_mtx(
    '/home/lixiangyu/zr/Annotate/data/self_data_human/CBT', 
    var_names='gene_symbols',             
    cache=True) 


In [30]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10176 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [31]:
# include dataset column
adata.obs["dataset"] = "CBT"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10176 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [32]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/CBT/output/CBT.h5ad")

In [33]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3327411/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [34]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/output_data/CBT/output/CBT.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/output_data/CBT/output/CBT_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Jul 31 18:10:21 2025 .. Analyzing all cells
Thu Jul 31 18:10:21 2025 .... Generating UMAP and estimating cell types
Thu Jul 31 18:10:48 2025 .... Estimating contamination
Thu Jul 31 18:10:53 2025 ...... Completed iteration: 10 | converge: 0.04491
Thu Jul 31 18:10:58 2025 ...... Completed iteration: 20 | converge: 0.01343
Thu Jul 31 18:11:02 2025 ...... Completed iteration: 30 | converge: 0.008185
Thu Jul 31 18:11:07 2025 ...... Completed iteration: 40 | converge: 0.005432
Thu Jul 31 18:11:12 2025 ...... Completed iteration: 50 | converge: 0.003978
Thu Jul 31 18:11:16 2025 ...... Completed iteration: 60 | converge: 0.003307
Thu Jul 31 18:11:21 2025 ...... Completed iteration: 70 | converge: 0.002781
Thu Jul 31 18:11:25 2025 ...... Completed iteration: 80 | converge: 0.002298
Thu Jul 31 18:11:30 2025 ...... Completed iteration: 90 | converge: 0.001907
Thu Jul 31 18:1

In [38]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/output_data/CBT/output/CBT_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10176 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [39]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [40]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 5000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10176
Number of cells after filter: 9827
Number of cells before MT filter: 9827
Number of cells after MT filter: 8464


In [41]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3327411/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8464 × 23395
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [42]:
adata.write("/home/lixiangyu/zr/Annotate/output_data/CBT/output/CBT_postQC.h5ad")

### Atherosclerosis, In-Stent Restenosis

#### IAISR////min_genes=200/// max_genes = 4000//min_cells=5

In [25]:
adata = sc.read_10x_mtx(
    './data/self_data_human/IAISR', 
    var_names='gene_symbols',             
    cache=True) 


In [26]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 21898 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [27]:
# include dataset column
adata.obs["dataset"] = "IAISR"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 21898 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [28]:
adata.write("./output_data/IAISR/output/IAISR.h5ad")

In [29]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2135878/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [30]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/IAISR/output/IAISR.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/IAISR/output/IAISR_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Mar 25 12:26:22 2025 .. Analyzing all cells
Tue Mar 25 12:26:22 2025 .... Generating UMAP and estimating cell types
Tue Mar 25 12:27:25 2025 .... Estimating contamination
Tue Mar 25 12:27:31 2025 ...... Completed iteration: 10 | converge: 0.0355
Tue Mar 25 12:27:37 2025 ...... Completed iteration: 20 | converge: 0.015
Tue Mar 25 12:27:42 2025 ...... Completed iteration: 30 | converge: 0.008828
Tue Mar 25 12:27:47 2025 ...... Completed iteration: 40 | converge: 0.005294
Tue Mar 25 12:27:53 2025 ...... Completed iteration: 50 | converge: 0.003709
Tue Mar 25 12:27:58 2025 ...... Completed iteration: 60 | converge: 0.002734
Tue Mar 25 12:28:04 2025 ...... Completed iteration: 70 | converge: 0.001958
Tue Mar 25 12:28:09 2025 ...... Completed iteration: 80 | converge: 0.001797
Tue Mar 25 12:28:15 2025 ...... Completed iteration: 90 | converge: 0.001626
Tue Mar 25 12:28:2

In [31]:
adata = sc.read_h5ad("./output_data/IAISR/output/IAISR_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 21898 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [32]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [33]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=5)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 5] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 21898
Number of cells after filter: 21687
Number of cells before MT filter: 21687
Number of cells after MT filter: 16220


In [34]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2135878/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 16220 × 20196
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [35]:
adata.write("./output_data/IAISR/output/IAISR_postQC.h5ad")

#### ISR-7-1

In [26]:
adata = sc.read_10x_mtx(
    './data/self_data_human/ISR-7-1', 
    var_names='gene_symbols',             
    cache=True) 


In [27]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10809 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [28]:
# include dataset column
adata.obs["dataset"] = "ISR-7-1"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10809 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [29]:
adata.write("./output_data/ISR-7-1/output/ISR-7-1.h5ad")

In [30]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3061073/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [31]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/ISR-7-1/output/ISR-7-1.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/ISR-7-1/output/ISR-7-1_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Mar 18 17:07:37 2025 .. Analyzing all cells
Tue Mar 18 17:07:37 2025 .... Generating UMAP and estimating cell types
Tue Mar 18 17:08:05 2025 .... Estimating contamination
Tue Mar 18 17:08:09 2025 ...... Completed iteration: 10 | converge: 0.02823
Tue Mar 18 17:08:12 2025 ...... Completed iteration: 20 | converge: 0.009181
Tue Mar 18 17:08:16 2025 ...... Completed iteration: 30 | converge: 0.005155
Tue Mar 18 17:08:20 2025 ...... Completed iteration: 40 | converge: 0.003081
Tue Mar 18 17:08:23 2025 ...... Completed iteration: 50 | converge: 0.002247
Tue Mar 18 17:08:27 2025 ...... Completed iteration: 60 | converge: 0.001577
Tue Mar 18 17:08:30 2025 ...... Completed iteration: 70 | converge: 0.001151
Tue Mar 18 17:08:34 2025 ...... Completed iteration: 80 | converge: 0.00137
Tue Mar 18 17:08:37 2025 ...... Completed iteration: 90 | converge: 0.0009732
Tue Mar 18 17:

In [32]:
adata = sc.read_h5ad("./output_data/ISR-7-1/output/ISR-7-1_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10809 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [33]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [34]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10809
Number of cells after filter: 10398
Number of cells before MT filter: 10398
Number of cells after MT filter: 8947


In [35]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3061073/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8947 × 20863
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [36]:
adata.write("./output_data/ISR-7-1/output/ISR-7-1_postQC.h5ad")

#### ISR-7-2

In [10]:
adata = sc.read_10x_mtx(
    './data/self_data_human/ISR-7-2', 
    var_names='gene_symbols',             
    cache=True) 


In [9]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 10328 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [39]:
# include dataset column
adata.obs["dataset"] = "ISR-7-2"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10328 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [40]:
adata.write("./output_data/ISR-7-2/output/ISR-7-2.h5ad")

In [41]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3061073/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [42]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/ISR-7-2/output/ISR-7-2.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/ISR-7-2/output/ISR-7-2_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Mar 18 17:12:04 2025 .. Analyzing all cells
Tue Mar 18 17:12:04 2025 .... Generating UMAP and estimating cell types
Tue Mar 18 17:12:36 2025 .... Estimating contamination
Tue Mar 18 17:12:41 2025 ...... Completed iteration: 10 | converge: 0.05202
Tue Mar 18 17:12:45 2025 ...... Completed iteration: 20 | converge: 0.01808
Tue Mar 18 17:12:49 2025 ...... Completed iteration: 30 | converge: 0.005873
Tue Mar 18 17:12:53 2025 ...... Completed iteration: 40 | converge: 0.00447
Tue Mar 18 17:12:57 2025 ...... Completed iteration: 50 | converge: 0.004314
Tue Mar 18 17:13:02 2025 ...... Completed iteration: 60 | converge: 0.003535
Tue Mar 18 17:13:06 2025 ...... Completed iteration: 70 | converge: 0.002812
Tue Mar 18 17:13:11 2025 ...... Completed iteration: 80 | converge: 0.002163
Tue Mar 18 17:13:15 2025 ...... Completed iteration: 90 | converge: 0.001872
Tue Mar 18 17:13

In [43]:
adata = sc.read_h5ad("./output_data/ISR-7-2/output/ISR-7-2_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10328 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [44]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [45]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10328
Number of cells after filter: 10089
Number of cells before MT filter: 10089
Number of cells after MT filter: 8944


In [46]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3061073/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8944 × 21434
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [47]:
adata.write("./output_data/ISR-7-2/output/ISR-7-2_postQC.h5ad")

#### ISR-8

In [28]:
adata = sc.read_10x_mtx(
    './data/self_data_human/ISR-8', 
    var_names='gene_symbols',             
    cache=True) 


In [29]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 5980 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [30]:
# include dataset column
adata.obs["dataset"] = "ISR-8"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 5980 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [31]:
adata.write("./output_data/ISR-8/output/ISR-8.h5ad")

In [32]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [33]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/ISR-8/output/ISR-8.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/ISR-8/output/ISR-8_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 11:09:49 2025 .. Analyzing all cells
Wed Mar 19 11:09:49 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 11:10:11 2025 .... Estimating contamination
Wed Mar 19 11:10:15 2025 ...... Completed iteration: 10 | converge: 0.01929
Wed Mar 19 11:10:18 2025 ...... Completed iteration: 20 | converge: 0.007831
Wed Mar 19 11:10:21 2025 ...... Completed iteration: 30 | converge: 0.004525
Wed Mar 19 11:10:24 2025 ...... Completed iteration: 40 | converge: 0.002971
Wed Mar 19 11:10:27 2025 ...... Completed iteration: 50 | converge: 0.00201
Wed Mar 19 11:10:30 2025 ...... Completed iteration: 60 | converge: 0.001775
Wed Mar 19 11:10:33 2025 ...... Completed iteration: 70 | converge: 0.001097
Wed Mar 19 11:10:35 2025 ...... Completed iteration: 75 | converge: 0.0009919
Wed Mar 19 11:10:35 2025 .. Calculating final decontaminated matrix
-----------------------

In [34]:
adata = sc.read_h5ad("./output_data/ISR-8/output/ISR-8_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 5980 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [35]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [36]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 5980
Number of cells after filter: 5029
Number of cells before MT filter: 5029
Number of cells after MT filter: 4305


In [37]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 4305 × 20388
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [38]:
adata.write("./output_data/ISR-8/output/ISR-8_postQC.h5ad")

#### ISR-9

In [69]:
adata = sc.read_10x_mtx(
    './data/self_data_human/ISR-9', 
    var_names='gene_symbols',             
    cache=True) 


In [70]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 14324 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [71]:
# include dataset column
adata.obs["dataset"] = "ISR-9"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 14324 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [72]:
adata.write("./output_data/ISR-9/output/ISR-9.h5ad")

In [73]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [74]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/ISR-9/output/ISR-9.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/ISR-9/output/ISR-9_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 11:51:07 2025 .. Analyzing all cells
Wed Mar 19 11:51:08 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 11:51:52 2025 .... Estimating contamination
Wed Mar 19 11:52:00 2025 ...... Completed iteration: 10 | converge: 0.03028
Wed Mar 19 11:52:07 2025 ...... Completed iteration: 20 | converge: 0.0101
Wed Mar 19 11:52:15 2025 ...... Completed iteration: 30 | converge: 0.005734
Wed Mar 19 11:52:22 2025 ...... Completed iteration: 40 | converge: 0.003386
Wed Mar 19 11:52:28 2025 ...... Completed iteration: 50 | converge: 0.002156
Wed Mar 19 11:52:36 2025 ...... Completed iteration: 60 | converge: 0.001683
Wed Mar 19 11:52:42 2025 ...... Completed iteration: 70 | converge: 0.001513
Wed Mar 19 11:52:50 2025 ...... Completed iteration: 80 | converge: 0.001086
Wed Mar 19 11:52:53 2025 ...... Completed iteration: 83 | converge: 0.0009934
Wed Mar 19 11:5

In [ ]:
adata = sc.read_h5ad("./output_data/ISR-9/output/ISR-9_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 14324 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [76]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 14324
Number of cells after filter: 13792
Number of cells before MT filter: 13792
Number of cells after MT filter: 13137


In [ ]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 13137 × 22041
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [79]:
adata.write("./output_data/ISR-9/output/ISR-9_postQC.h5ad")

#### LCH_matrix <20

In [11]:
adata = sc.read_10x_mtx(
    './data/self_data_human/LCH_matrix', 
    var_names='gene_symbols',             
    cache=True) 


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/__init__.py:55: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [12]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 6453 × 29317
    obs: 'sample'
    var: 'gene_ids'

In [13]:
# include dataset column
adata.obs["dataset"] = "LCH_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 6453 × 29317
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [14]:
adata.write("./output_data/LCH_matrix/output/LCH_matrix.h5ad")

In [15]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3803219/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [16]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/LCH_matrix/output/LCH_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/LCH_matrix/output/LCH_matrix_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Sun Apr  6 18:48:27 2025 .. Analyzing all cells
Sun Apr  6 18:48:27 2025 .... Generating UMAP and estimating cell types
Sun Apr  6 18:48:54 2025 .... Estimating contamination
Sun Apr  6 18:48:58 2025 ...... Completed iteration: 10 | converge: 0.01989
Sun Apr  6 18:49:01 2025 ...... Completed iteration: 20 | converge: 0.006732
Sun Apr  6 18:49:04 2025 ...... Completed iteration: 30 | converge: 0.003463
Sun Apr  6 18:49:06 2025 ...... Completed iteration: 40 | converge: 0.001882
Sun Apr  6 18:49:09 2025 ...... Completed iteration: 50 | converge: 0.001088
Sun Apr  6 18:49:10 2025 ...... Completed iteration: 52 | converge: 0.0009775
Sun Apr  6 18:49:10 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 45.57821 secs
--------------------------------------------------


In [17]:
adata = sc.read_h5ad("./output_data/LCH_matrix/output/LCH_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 6453 × 29317
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [18]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [19]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 20] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 6453
Number of cells after filter: 6351
Number of cells before MT filter: 6351
Number of cells after MT filter: 5710


In [20]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3803219/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 5710 × 22758
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [21]:
adata.write("./output_data/LCH_matrix/output/LCH_matrix_postQC.h5ad")

#### LHJ20230414_matrix <20

In [22]:
adata = sc.read_10x_mtx(
    './data/self_data_human/LHJ20230414_matrix', 
    var_names='gene_symbols',             
    cache=True) 


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/__init__.py:55: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [23]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 7420 × 58336
    obs: 'sample'
    var: 'gene_ids'

In [24]:
# include dataset column
adata.obs["dataset"] = "LHJ20230414_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 7420 × 58336
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [25]:
adata.write("./output_data/LHJ20230414_matrix/output/LHJ20230414_matrix.h5ad")

In [26]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3803219/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [27]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/LHJ20230414_matrix/output/LHJ20230414_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/LHJ20230414_matrix/output/LHJ20230414_matrix_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Sun Apr  6 18:52:00 2025 .. Analyzing all cells
Sun Apr  6 18:52:00 2025 .... Generating UMAP and estimating cell types
Sun Apr  6 18:52:31 2025 .... Estimating contamination
Sun Apr  6 18:52:37 2025 ...... Completed iteration: 10 | converge: 0.03193
Sun Apr  6 18:52:41 2025 ...... Completed iteration: 20 | converge: 0.01431
Sun Apr  6 18:52:45 2025 ...... Completed iteration: 30 | converge: 0.009676
Sun Apr  6 18:52:50 2025 ...... Completed iteration: 40 | converge: 0.006376
Sun Apr  6 18:52:54 2025 ...... Completed iteration: 50 | converge: 0.004931
Sun Apr  6 18:52:59 2025 ...... Completed iteration: 60 | converge: 0.003736
Sun Apr  6 18:53:03 2025 ...... Completed iteration: 70 | converge: 0.003096
Sun Apr  6 18:53:07 2025 ...... Completed iteration: 80 | converge: 0.002612
Sun Apr  6 18:53:11 2025 ...... Completed iteration: 90 | converge: 0.002201
Sun Apr  6 18:5

In [28]:
adata = sc.read_h5ad("./output_data/LHJ20230414_matrix/output/LHJ20230414_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 7420 × 58336
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [29]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [30]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 20] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 7420
Number of cells after filter: 7126
Number of cells before MT filter: 7126
Number of cells after MT filter: 6187


In [31]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3803219/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 6187 × 22277
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [32]:
adata.write("./output_data/LHJ20230414_matrix/output/LHJ20230414_matrix_postQC.h5ad")

#### LW_matrix max_genes = 40000;min_genes=20;min_cells=1;<15

In [3]:
adata = sc.read_10x_mtx(
    './data/self_data_human/LW_matrix', 
    var_names='gene_symbols',             
    cache=True) 


In [4]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 6012 × 58336
    obs: 'sample'
    var: 'gene_ids'

In [5]:
# include dataset column
adata.obs["dataset"] = "LW_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 6012 × 58336
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [6]:
adata.write("./output_data/LW_matrix/output/LW_matrix.h5ad")

In [7]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/numpy2ri.py:252: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_1457909/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [8]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/LW_matrix/output/LW_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/LW_matrix/output/LW_matrix_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [9]:
adata = sc.read_h5ad("./output_data/LW_matrix/output/LW_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 6012 × 58336
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [10]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [11]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=20)  
sc.pp.filter_cells(adata, max_genes = 40000)  
sc.pp.filter_genes(adata, min_cells=1)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 6012
Number of cells after filter: 6012
Number of cells before MT filter: 6012
Number of cells after MT filter: 4784


In [12]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_1457909/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 4784 × 28158
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [13]:
adata.write("./output_data/LW_matrix/output/LW_matrix_postQC.h5ad")

#### POP-ISR-A

In [113]:
adata = sc.read_10x_mtx(
    './data/self_data_human/POP-ISR-A', 
    var_names='gene_symbols',             
    cache=True) 


In [114]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 12244 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [115]:
# include dataset column
adata.obs["dataset"] = "POP-ISR-A"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 12244 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [116]:
adata.write("./output_data/POP-ISR-A/output/POP-ISR-A.h5ad")

In [117]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [118]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/POP-ISR-A/output/POP-ISR-A.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/POP-ISR-A/output/POP-ISR-A_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 12:04:19 2025 .. Analyzing all cells
Wed Mar 19 12:04:19 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 12:04:56 2025 .... Estimating contamination
Wed Mar 19 12:05:02 2025 ...... Completed iteration: 10 | converge: 0.03161
Wed Mar 19 12:05:08 2025 ...... Completed iteration: 20 | converge: 0.01052
Wed Mar 19 12:05:14 2025 ...... Completed iteration: 30 | converge: 0.005405
Wed Mar 19 12:05:20 2025 ...... Completed iteration: 40 | converge: 0.003449
Wed Mar 19 12:05:26 2025 ...... Completed iteration: 50 | converge: 0.002336
Wed Mar 19 12:05:32 2025 ...... Completed iteration: 60 | converge: 0.00158
Wed Mar 19 12:05:38 2025 ...... Completed iteration: 70 | converge: 0.001093
Wed Mar 19 12:05:42 2025 ...... Completed iteration: 77 | converge: 0.000899
Wed Mar 19 12:05:42 2025 .. Calculating final decontaminated matrix
-------------------------

In [119]:
adata = sc.read_h5ad("./output_data/POP-ISR-A/output/POP-ISR-A_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 12244 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [120]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [121]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 12244
Number of cells after filter: 11554
Number of cells before MT filter: 11554
Number of cells after MT filter: 10168


In [122]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 10168 × 22239
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [123]:
adata.write("./output_data/POP-ISR-A/output/POP-ISR-A_postQC.h5ad")

#### POP-ISR2-B

In [124]:
adata = sc.read_10x_mtx(
    './data/self_data_human/POP-ISR2-B', 
    var_names='gene_symbols',             
    cache=True) 


In [125]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 12224 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [126]:
# include dataset column
adata.obs["dataset"] = "POP-ISR2-B"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 12224 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [127]:
adata.write("./output_data/POP-ISR2-B/output/POP-ISR2-B.h5ad")

In [128]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [129]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/POP-ISR2-B/output/POP-ISR2-B.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/POP-ISR2-B/output/POP-ISR2-B_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 12:08:06 2025 .. Analyzing all cells
Wed Mar 19 12:08:07 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 12:08:48 2025 .... Estimating contamination
Wed Mar 19 12:08:54 2025 ...... Completed iteration: 10 | converge: 0.03592
Wed Mar 19 12:09:00 2025 ...... Completed iteration: 20 | converge: 0.01253
Wed Mar 19 12:09:06 2025 ...... Completed iteration: 30 | converge: 0.00618
Wed Mar 19 12:09:11 2025 ...... Completed iteration: 40 | converge: 0.004295
Wed Mar 19 12:09:16 2025 ...... Completed iteration: 50 | converge: 0.003149
Wed Mar 19 12:09:21 2025 ...... Completed iteration: 60 | converge: 0.0023
Wed Mar 19 12:09:26 2025 ...... Completed iteration: 70 | converge: 0.001835
Wed Mar 19 12:09:32 2025 ...... Completed iteration: 80 | converge: 0.001387
Wed Mar 19 12:09:38 2025 ...... Completed iteration: 90 | converge: 0.001104
Wed Mar 19 12:09:4

In [130]:
adata = sc.read_h5ad("./output_data/POP-ISR2-B/output/POP-ISR2-B_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 12224 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [131]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [132]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 12224
Number of cells after filter: 11909
Number of cells before MT filter: 11909
Number of cells after MT filter: 10040


In [133]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 10040 × 21841
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [134]:
adata.write("./output_data/POP-ISR2-B/output/POP-ISR2-B_postQC.h5ad")

#### S221031_matrix <20

In [33]:
adata = sc.read_10x_mtx(
    './data/self_data_human/S221031_matrix', 
    var_names='gene_symbols',             
    cache=True) 


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/__init__.py:55: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [34]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 10831 × 58336
    obs: 'sample'
    var: 'gene_ids'

In [35]:
# include dataset column
adata.obs["dataset"] = "S221031_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 10831 × 58336
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [36]:
adata.write("./output_data/S221031_matrix/output/S221031_matrix.h5ad")

In [37]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3803219/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [38]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/S221031_matrix/output/S221031_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/S221031_matrix/output/S221031_matrix_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Sun Apr  6 18:56:09 2025 .. Analyzing all cells
Sun Apr  6 18:56:09 2025 .... Generating UMAP and estimating cell types
Sun Apr  6 18:56:40 2025 .... Estimating contamination
Sun Apr  6 18:56:46 2025 ...... Completed iteration: 10 | converge: 0.03082
Sun Apr  6 18:56:51 2025 ...... Completed iteration: 20 | converge: 0.01493
Sun Apr  6 18:56:56 2025 ...... Completed iteration: 30 | converge: 0.01123
Sun Apr  6 18:57:01 2025 ...... Completed iteration: 40 | converge: 0.007831
Sun Apr  6 18:57:06 2025 ...... Completed iteration: 50 | converge: 0.005278
Sun Apr  6 18:57:11 2025 ...... Completed iteration: 60 | converge: 0.003838
Sun Apr  6 18:57:16 2025 ...... Completed iteration: 70 | converge: 0.0034
Sun Apr  6 18:57:21 2025 ...... Completed iteration: 80 | converge: 0.002785
Sun Apr  6 18:57:26 2025 ...... Completed iteration: 90 | converge: 0.002312
Sun Apr  6 18:57:3

In [39]:
adata = sc.read_h5ad("./output_data/S221031_matrix/output/S221031_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 10831 × 58336
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [40]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [41]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 20] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 10831
Number of cells after filter: 10794
Number of cells before MT filter: 10794
Number of cells after MT filter: 9863


In [42]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3803219/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 9863 × 23359
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [43]:
adata.write("./output_data/S221031_matrix/output/S221031_matrix_postQC.h5ad")

#### ZNB_matrix <15

In [36]:
adata = sc.read_10x_mtx(
    './data/self_data_human/ZNB_matrix', 
    var_names='gene_symbols',             
    cache=True) 


In [37]:
##without sampleID
adata.obs["sample"] = "sample"
adata

AnnData object with n_obs × n_vars = 8762 × 31061
    obs: 'sample'
    var: 'gene_ids'

In [38]:
# include dataset column
adata.obs["dataset"] = "ZNB_matrix"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8762 × 31061
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids'

In [39]:
adata.write("./output_data/ZNB_matrix/output/ZNB_matrix.h5ad")

In [40]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_2135878/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [41]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/ZNB_matrix/output/ZNB_matrix.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/ZNB_matrix/output/ZNB_matrix_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Mar 25 13:56:14 2025 .. Analyzing all cells
Tue Mar 25 13:56:15 2025 .... Generating UMAP and estimating cell types
Tue Mar 25 13:56:42 2025 .... Estimating contamination
Tue Mar 25 13:56:47 2025 ...... Completed iteration: 10 | converge: 0.02979
Tue Mar 25 13:56:50 2025 ...... Completed iteration: 20 | converge: 0.01235
Tue Mar 25 13:56:54 2025 ...... Completed iteration: 30 | converge: 0.008851
Tue Mar 25 13:56:57 2025 ...... Completed iteration: 40 | converge: 0.006282
Tue Mar 25 13:57:01 2025 ...... Completed iteration: 50 | converge: 0.004782
Tue Mar 25 13:57:05 2025 ...... Completed iteration: 60 | converge: 0.00371
Tue Mar 25 13:57:08 2025 ...... Completed iteration: 70 | converge: 0.003015
Tue Mar 25 13:57:12 2025 ...... Completed iteration: 80 | converge: 0.002392
Tue Mar 25 13:57:15 2025 ...... Completed iteration: 90 | converge: 0.00184
Tue Mar 25 13:57:

In [42]:
adata = sc.read_h5ad("./output_data/ZNB_matrix/output/ZNB_matrix_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8762 × 31061
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [43]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [44]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 15] ##10:4208
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8762
Number of cells after filter: 8601
Number of cells before MT filter: 8601
Number of cells after MT filter: 7393


In [45]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_2135878/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 7393 × 24203
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [46]:
adata.write("./output_data/ZNB_matrix/output/ZNB_matrix_postQC.h5ad")

### Thoracic Aortic Aneurys (TAA)

#### TAA---question:data

In [3]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA', 
    var_names='gene_symbols',             
    cache=True) 


error: Error -3 while decompressing data: invalid block type

In [ ]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

In [ ]:
# include dataset column
adata.obs["dataset"] = "TAA"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

In [ ]:
adata.write("./output_data/TAA/output/TAA.h5ad")

In [ ]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA/output/TAA.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA/output/TAA_postR.h5ad")

In [ ]:
adata = sc.read_h5ad("./output_data/TAA/output/TAA_postR.h5ad")
adata

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

In [ ]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

In [ ]:
adata.write("./output_data/TAA/output/TAA_postQC.h5ad")

#### TAA-A1-3-3LIB

In [186]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA-A1-3-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [187]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 14075 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [188]:
# include dataset column
adata.obs["dataset"] = "TAA-A1-3-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 14075 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [189]:
adata.write("./output_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB.h5ad")

In [190]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [191]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:01:29 2025 .. Analyzing all cells
Wed Mar 19 13:01:29 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:02:16 2025 .... Estimating contamination
Wed Mar 19 13:02:22 2025 ...... Completed iteration: 10 | converge: 0.02781
Wed Mar 19 13:02:29 2025 ...... Completed iteration: 20 | converge: 0.007915
Wed Mar 19 13:02:35 2025 ...... Completed iteration: 30 | converge: 0.004095
Wed Mar 19 13:02:42 2025 ...... Completed iteration: 40 | converge: 0.002836
Wed Mar 19 13:02:47 2025 ...... Completed iteration: 50 | converge: 0.001908
Wed Mar 19 13:02:53 2025 ...... Completed iteration: 60 | converge: 0.001007
Wed Mar 19 13:02:54 2025 ...... Completed iteration: 61 | converge: 0.0009651
Wed Mar 19 13:02:54 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 1.517559 mins
---

In [192]:
adata = sc.read_h5ad("./output_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 14075 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [193]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [194]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 14075
Number of cells after filter: 13590
Number of cells before MT filter: 13590
Number of cells after MT filter: 11756


In [195]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 11756 × 21144
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [196]:
adata.write("./output_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB_postQC.h5ad")

#### TAA-A1-5-3LIB

In [197]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA-A1-5-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [198]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 12520 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [199]:
# include dataset column
adata.obs["dataset"] = "TAA-A1-5-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 12520 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [200]:
adata.write("./output_data/TAA-A1-5-3LIB/output/TAA-A1-5-3LIB.h5ad")

In [201]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [202]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA-A1-5-3LIB/output/TAA-A1-5-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA-A1-5-3LIB/output/TAA-A1-5-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:06:15 2025 .. Analyzing all cells
Wed Mar 19 13:06:15 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:06:52 2025 .... Estimating contamination
Wed Mar 19 13:06:59 2025 ...... Completed iteration: 10 | converge: 0.04018
Wed Mar 19 13:07:05 2025 ...... Completed iteration: 20 | converge: 0.01514
Wed Mar 19 13:07:11 2025 ...... Completed iteration: 30 | converge: 0.007606
Wed Mar 19 13:07:15 2025 ...... Completed iteration: 40 | converge: 0.005102
Wed Mar 19 13:07:20 2025 ...... Completed iteration: 50 | converge: 0.003521
Wed Mar 19 13:07:26 2025 ...... Completed iteration: 60 | converge: 0.002806
Wed Mar 19 13:07:32 2025 ...... Completed iteration: 70 | converge: 0.001893
Wed Mar 19 13:07:37 2025 ...... Completed iteration: 80 | converge: 0.001408
Wed Mar 19 13:07:41 2025 ...... Completed iteration: 90 | converge: 0.001193
Wed Mar 19 13:0

In [203]:
adata = sc.read_h5ad("./output_data/TAA-A1-5-3LIB/output/TAA-A1-5-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 12520 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [204]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [205]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 12520
Number of cells after filter: 12259
Number of cells before MT filter: 12259
Number of cells after MT filter: 11523


In [206]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 11523 × 20981
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [207]:
adata.write("./output_data/TAA-A1-5-3LIB/output/TAA-A1-5-3LIB_postQC.h5ad")

#### TAA-B1-5-3LIB

In [208]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA-B1-5-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [209]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 5070 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [210]:
# include dataset column
adata.obs["dataset"] = "TAA-B1-5-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 5070 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [211]:
adata.write("./output_data/TAA-B1-5-3LIB/output/TAA-B1-5-3LIB.h5ad")

In [212]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [213]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA-B1-5-3LIB/output/TAA-B1-5-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA-B1-5-3LIB/output/TAA-B1-5-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:09:11 2025 .. Analyzing all cells
Wed Mar 19 13:09:11 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:09:33 2025 .... Estimating contamination
Wed Mar 19 13:09:35 2025 ...... Completed iteration: 10 | converge: 0.02723
Wed Mar 19 13:09:38 2025 ...... Completed iteration: 20 | converge: 0.01489
Wed Mar 19 13:09:40 2025 ...... Completed iteration: 30 | converge: 0.007906
Wed Mar 19 13:09:43 2025 ...... Completed iteration: 40 | converge: 0.005303
Wed Mar 19 13:09:46 2025 ...... Completed iteration: 50 | converge: 0.004017
Wed Mar 19 13:09:50 2025 ...... Completed iteration: 60 | converge: 0.003177
Wed Mar 19 13:09:54 2025 ...... Completed iteration: 70 | converge: 0.002612
Wed Mar 19 13:09:56 2025 ...... Completed iteration: 80 | converge: 0.002194
Wed Mar 19 13:09:58 2025 ...... Completed iteration: 90 | converge: 0.001845
Wed Mar 19 13:1

In [214]:
adata = sc.read_h5ad("./output_data/TAA-B1-5-3LIB/output/TAA-B1-5-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 5070 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [215]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [216]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 5070
Number of cells after filter: 4715
Number of cells before MT filter: 4715
Number of cells after MT filter: 4464


In [217]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 4464 × 20501
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [218]:
adata.write("./output_data/TAA-B1-5-3LIB/output/TAA-B1-5-3LIB_postQC.h5ad")

#### TAA-Z1

In [219]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA-Z1', 
    var_names='gene_symbols',             
    cache=True) 


In [220]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 4624 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [221]:
# include dataset column
adata.obs["dataset"] = "TAA-Z1"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 4624 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [222]:
adata.write("./output_data/TAA-Z1/output/TAA-Z1.h5ad")

In [223]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [224]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA-Z1/output/TAA-Z1.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA-Z1/output/TAA-Z1_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:11:38 2025 .. Analyzing all cells
Wed Mar 19 13:11:38 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:11:58 2025 .... Estimating contamination
Wed Mar 19 13:12:01 2025 ...... Completed iteration: 10 | converge: 0.02156
Wed Mar 19 13:12:04 2025 ...... Completed iteration: 20 | converge: 0.01025
Wed Mar 19 13:12:06 2025 ...... Completed iteration: 30 | converge: 0.006476
Wed Mar 19 13:12:09 2025 ...... Completed iteration: 40 | converge: 0.004552
Wed Mar 19 13:12:11 2025 ...... Completed iteration: 50 | converge: 0.003552
Wed Mar 19 13:12:13 2025 ...... Completed iteration: 60 | converge: 0.00235
Wed Mar 19 13:12:15 2025 ...... Completed iteration: 70 | converge: 0.002061
Wed Mar 19 13:12:18 2025 ...... Completed iteration: 80 | converge: 0.001582
Wed Mar 19 13:12:20 2025 ...... Completed iteration: 90 | converge: 0.001526
Wed Mar 19 13:12

In [225]:
adata = sc.read_h5ad("./output_data/TAA-Z1/output/TAA-Z1_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 4624 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [226]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [227]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 4624
Number of cells after filter: 4002
Number of cells before MT filter: 4002
Number of cells after MT filter: 3393


In [228]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 3393 × 18960
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [229]:
adata.write("./output_data/TAA-Z1/output/TAA-Z1_postQC.h5ad")

#### TAA-Z3

In [230]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAA-Z3', 
    var_names='gene_symbols',             
    cache=True) 


In [231]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 3126 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [232]:
# include dataset column
adata.obs["dataset"] = "TAA-Z3"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 3126 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [233]:
adata.write("./output_data/TAA-Z3/output/TAA-Z3.h5ad")

In [234]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [235]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAA-Z3/output/TAA-Z3.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAA-Z3/output/TAA-Z3_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:13:41 2025 .. Analyzing all cells
Wed Mar 19 13:13:41 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:13:58 2025 .... Estimating contamination
Wed Mar 19 13:13:59 2025 ...... Completed iteration: 10 | converge: 0.01575
Wed Mar 19 13:13:59 2025 ...... Completed iteration: 20 | converge: 0.006316
Wed Mar 19 13:14:00 2025 ...... Completed iteration: 30 | converge: 0.002874
Wed Mar 19 13:14:01 2025 ...... Completed iteration: 40 | converge: 0.002056
Wed Mar 19 13:14:02 2025 ...... Completed iteration: 50 | converge: 0.001386
Wed Mar 19 13:14:02 2025 ...... Completed iteration: 60 | converge: 0.001268
Wed Mar 19 13:14:03 2025 ...... Completed iteration: 70 | converge: 0.0009854
Wed Mar 19 13:14:03 2025 .. Calculating final decontaminated matrix
--------------------------------------------------
Completed DecontX. Total time: 23.0914 secs
----

In [236]:
adata = sc.read_h5ad("./output_data/TAA-Z3/output/TAA-Z3_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 3126 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [237]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [238]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 3126
Number of cells after filter: 3113
Number of cells before MT filter: 3113
Number of cells after MT filter: 3075


In [239]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 3075 × 15057
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [240]:
adata.write("./output_data/TAA-Z3/output/TAA-Z3_postQC.h5ad")

#### TAD-AD1-2-3LIB

In [241]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAD-AD1-2-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [242]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 15887 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [243]:
# include dataset column
adata.obs["dataset"] = "TAD-AD1-2-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 15887 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [244]:
adata.write("./output_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB.h5ad")

In [245]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3304628/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [246]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 13:17:50 2025 .. Analyzing all cells
Wed Mar 19 13:17:50 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 13:18:40 2025 .... Estimating contamination
Wed Mar 19 13:18:47 2025 ...... Completed iteration: 10 | converge: 0.02812
Wed Mar 19 13:18:52 2025 ...... Completed iteration: 20 | converge: 0.009776
Wed Mar 19 13:18:57 2025 ...... Completed iteration: 30 | converge: 0.00519
Wed Mar 19 13:19:01 2025 ...... Completed iteration: 40 | converge: 0.003435
Wed Mar 19 13:19:07 2025 ...... Completed iteration: 50 | converge: 0.002367
Wed Mar 19 13:19:12 2025 ...... Completed iteration: 60 | converge: 0.001829
Wed Mar 19 13:19:16 2025 ...... Completed iteration: 70 | converge: 0.001664
Wed Mar 19 13:19:21 2025 ...... Completed iteration: 80 | converge: 0.001198
Wed Mar 19 13:19:22 2025 ...... Completed iteration: 83 | converge: 0.0009966
Wed Mar 19 13:

In [247]:
adata = sc.read_h5ad("./output_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 15887 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [248]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [249]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 15887
Number of cells after filter: 15848
Number of cells before MT filter: 15848
Number of cells after MT filter: 15142


In [250]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3304628/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 15142 × 20963
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [251]:
adata.write("./output_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB_postQC.h5ad")

#### TAD1-Z3-3LIB

In [4]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAD1-Z3-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [5]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8912 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [6]:
# include dataset column
adata.obs["dataset"] = "TAD1-Z3-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8912 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [7]:
adata.write("./output_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB.h5ad")

In [8]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/numpy2ri.py:252: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3574098/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [9]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [10]:
adata = sc.read_h5ad("./output_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8912 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [11]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [12]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8912
Number of cells after filter: 8607
Number of cells before MT filter: 8607
Number of cells after MT filter: 8039


In [13]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3574098/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 8039 × 20060
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [14]:
adata.write("./output_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB_postQC.h5ad")

#### TAD2-Z1-3LIB

In [15]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAD2-Z1-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [16]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 1711 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [17]:
# include dataset column
adata.obs["dataset"] = "TAD2-Z1-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 1711 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [18]:
adata.write("./output_data/TAD2-Z1-3LIB/output/TAD2-Z1-3LIB.h5ad")

In [19]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3574098/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [20]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAD2-Z1-3LIB/output/TAD2-Z1-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAD2-Z1-3LIB/output/TAD2-Z1-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 20:34:09 2025 .. Analyzing all cells
Wed Mar 19 20:34:09 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 20:34:18 2025 .... Estimating contamination
Wed Mar 19 20:34:19 2025 ...... Completed iteration: 10 | converge: 0.02483
Wed Mar 19 20:34:20 2025 ...... Completed iteration: 20 | converge: 0.01095
Wed Mar 19 20:34:21 2025 ...... Completed iteration: 30 | converge: 0.005827
Wed Mar 19 20:34:21 2025 ...... Completed iteration: 40 | converge: 0.002975
Wed Mar 19 20:34:22 2025 ...... Completed iteration: 50 | converge: 0.001721
Wed Mar 19 20:34:22 2025 ...... Completed iteration: 60 | converge: 0.001207
Wed Mar 19 20:34:23 2025 ...... Completed iteration: 70 | converge: 0.001438
Wed Mar 19 20:34:24 2025 ...... Completed iteration: 80 | converge: 0.0009822
Wed Mar 19 20:34:24 2025 .. Calculating final decontaminated matrix
-----------------------

In [21]:
adata = sc.read_h5ad("./output_data/TAD2-Z1-3LIB/output/TAD2-Z1-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 1711 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [22]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [23]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 1711
Number of cells after filter: 1666
Number of cells before MT filter: 1666
Number of cells after MT filter: 1493


In [24]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3574098/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 1493 × 16344
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [25]:
adata.write("./output_data/TAD2-Z1-3LIB/output/TAD2-Z1-3LIB_postQC.h5ad")

#### TAD2-Z2-3LIB

In [26]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAD2-Z2-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [27]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 14038 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [28]:
# include dataset column
adata.obs["dataset"] = "TAD2-Z2-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 14038 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [29]:
adata.write("./output_data/TAD2-Z2-3LIB/output/TAD2-Z2-3LIB.h5ad")

In [30]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3574098/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [31]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAD2-Z2-3LIB/output/TAD2-Z2-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAD2-Z2-3LIB/output/TAD2-Z2-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 20:37:40 2025 .. Analyzing all cells
Wed Mar 19 20:37:40 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 20:38:15 2025 .... Estimating contamination
Wed Mar 19 20:38:19 2025 ...... Completed iteration: 10 | converge: 0.03082
Wed Mar 19 20:38:23 2025 ...... Completed iteration: 20 | converge: 0.01272
Wed Mar 19 20:38:27 2025 ...... Completed iteration: 30 | converge: 0.008009
Wed Mar 19 20:38:31 2025 ...... Completed iteration: 40 | converge: 0.005021
Wed Mar 19 20:38:35 2025 ...... Completed iteration: 50 | converge: 0.003826
Wed Mar 19 20:38:39 2025 ...... Completed iteration: 60 | converge: 0.003012
Wed Mar 19 20:38:43 2025 ...... Completed iteration: 70 | converge: 0.002527
Wed Mar 19 20:38:47 2025 ...... Completed iteration: 80 | converge: 0.002164
Wed Mar 19 20:38:51 2025 ...... Completed iteration: 90 | converge: 0.001863
Wed Mar 19 20:3

In [32]:
adata = sc.read_h5ad("./output_data/TAD2-Z2-3LIB/output/TAD2-Z2-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 14038 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [33]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [34]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 14038
Number of cells after filter: 13890
Number of cells before MT filter: 13890
Number of cells after MT filter: 13392


In [35]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3574098/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 13392 × 20119
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [36]:
adata.write("./output_data/TAD2-Z2-3LIB/output/TAD2-Z2-3LIB_postQC.h5ad")

#### TAD2-Z3-3LIB

In [37]:
adata = sc.read_10x_mtx(
    './data/self_data_human/TAD2-Z3-3LIB', 
    var_names='gene_symbols',             
    cache=True) 


In [38]:
# new .obs["sample"] column with sample id from barcode suffix
adata.obs["sample"] = [barcode.split("-")[1] for barcode in adata.obs.index]
adata

AnnData object with n_obs × n_vars = 8523 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types'

In [39]:
# include dataset column
adata.obs["dataset"] = "TAD2-Z3-3LIB"
# include symptom column
adata.obs["symptoms"] = 'not stated'
adata

AnnData object with n_obs × n_vars = 8523 × 36601
    obs: 'sample', 'dataset', 'symptoms'
    var: 'gene_ids', 'feature_types'

In [40]:
adata.write("./output_data/TAD2-Z3-3LIB/output/TAD2-Z3-3LIB.h5ad")

In [41]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/robjects/pandas2ri.py:368: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  warnings.warn('The global conversion available with activate() '
/tmp/ipykernel_3574098/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [42]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_data/TAD2-Z3-3LIB/output/TAD2-Z3-3LIB.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_data/TAD2-Z3-3LIB/output/TAD2-Z3-3LIB_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/rpy2/ipython/rmagic.py:990: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  displaypub.publish_display_data(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Wed Mar 19 20:42:28 2025 .. Analyzing all cells
Wed Mar 19 20:42:28 2025 .... Generating UMAP and estimating cell types
Wed Mar 19 20:42:56 2025 .... Estimating contamination
Wed Mar 19 20:43:00 2025 ...... Completed iteration: 10 | converge: 0.02042
Wed Mar 19 20:43:03 2025 ...... Completed iteration: 20 | converge: 0.005446
Wed Mar 19 20:43:06 2025 ...... Completed iteration: 30 | converge: 0.003543
Wed Mar 19 20:43:09 2025 ...... Completed iteration: 40 | converge: 0.00245
Wed Mar 19 20:43:12 2025 ...... Completed iteration: 50 | converge: 0.001787
Wed Mar 19 20:43:15 2025 ...... Completed iteration: 60 | converge: 0.001305
Wed Mar 19 20:43:19 2025 ...... Completed iteration: 70 | converge: 0.001069
Wed Mar 19 20:43:19 2025 ...... Completed iteration: 72 | converge: 0.0009894
Wed Mar 19 20:43:19 2025 .. Calculating final decontaminated matrix
-----------------------

In [43]:
adata = sc.read_h5ad("./output_data/TAD2-Z3-3LIB/output/TAD2-Z3-3LIB_postR.h5ad")
adata

AnnData object with n_obs × n_vars = 8523 × 36601
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'gene_ids', 'feature_types'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'decontXcounts'

In [44]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [45]:
print('Number of cells before filter: {:d}'.format(adata.n_obs))

sc.pp.filter_cells(adata, min_genes=200)  
sc.pp.filter_cells(adata, max_genes = 4000)  
sc.pp.filter_genes(adata, min_cells=3)
print('Number of cells after filter: {:d}'.format(adata.n_obs))

print('Number of cells before MT filter: {:d}'.format(adata.n_obs))

adata = adata[adata.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))

Number of cells before filter: 8523
Number of cells after filter: 8138
Number of cells before MT filter: 8138
Number of cells after MT filter: 7614


In [46]:
adata.layers["uncorrected_counts"] = adata.X.copy()
adata.layers["raw_decontXcounts"] = adata.layers["decontXcounts"].copy()
adata.X = np.around(adata.layers["raw_decontXcounts"].copy()).astype(int)
del adata.layers["decontXcounts"]
adata

/tmp/ipykernel_3574098/191121921.py:1: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata.layers["uncorrected_counts"] = adata.X.copy()


AnnData object with n_obs × n_vars = 7614 × 20006
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [47]:
adata.write("./output_data/TAD2-Z3-3LIB/output/TAD2-Z3-3LIB_postQC.h5ad")